# Import necessary modules

In [15]:
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

# Load and preprocess MNIST dataset

In [16]:
(trainX, trainY), (testX, testY) = mnist.load_data()
trainX, testX = trainX/255.0, testX/255.0
trainX = trainX.reshape(-1, 784)
testX = testX.reshape(-1, 784)
trainY = to_categorical(trainY, 10)
testY = to_categorical(testY, 10)

In [17]:
trainX.shape, trainY.shape, testX.shape, testY.shape

((60000, 784), (60000, 10), (10000, 784), (10000, 10))

# Build the model

In [18]:
class MLP(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.d1 = tf.keras.layers.Dense(8, activation='relu')
        self.d2 = tf.keras.layers.Dense(4, activation='relu')
        self.d3 = tf.keras.layers.Dense(4, activation='relu')
        self.out = tf.keras.layers.Dense(10, activation='softmax')

    def call(self, x):
        x = self.d1(x)
        x = self.d2(x)
        x = self.d3(x)
        return self.out(x)

gradient_tape_model = MLP()

# Loss function and optimizer

In [19]:
loss_function = tf.keras.losses.CategoricalCrossentropy(from_logits=False)
optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)

# Training loop (using gradient tape)

In [20]:
EPOCHS = 10
BATCH_SIZE = 128

train_test_split = int(0.8 * len(trainX))
# Corrected slicing for train and validation sets
trainX_split = trainX[:train_test_split]
trainY_split = trainY[:train_test_split]
valX = trainX[train_test_split:]
valY = trainY[train_test_split:]

# Create training dataset
train_ds = tf.data.Dataset.from_tensor_slices((trainX_split, trainY_split))
train_ds = train_ds.shuffle(buffer_size=10000).batch(BATCH_SIZE)

# Create validation dataset
val_ds = tf.data.Dataset.from_tensor_slices((valX, valY))
val_ds = val_ds.batch(BATCH_SIZE)

for epoch in range(EPOCHS):
    # Training loop
    for x_batch, y_batch in train_ds:
        with tf.GradientTape() as tape:
            probs = gradient_tape_model(x_batch)
            loss = loss_function(y_batch, probs)
        grads = tape.gradient(loss, gradient_tape_model.trainable_variables)
        optimizer.apply_gradients(zip(grads, gradient_tape_model.trainable_variables))

    # Validation loop
    val_losses = []
    val_accuracy = tf.keras.metrics.CategoricalAccuracy()

    for x_val_batch, y_val_batch in val_ds:
        val_probs = gradient_tape_model(x_val_batch)
        val_loss = loss_function(y_val_batch, val_probs)
        val_losses.append(val_loss.numpy())
        val_accuracy.update_state(y_val_batch, val_probs)

    val_loss_avg = sum(val_losses) / len(val_losses)

    print(f"Epoch {epoch+1}: Training Loss = {loss.numpy():.4f}, "
          f"Validation Loss = {val_loss_avg:.4f}, "
          f"Validation Accuracy = {val_accuracy.result().numpy():.4f}")

Epoch 1: Training Loss = 0.8466, Validation Loss = 0.5612, Validation Accuracy = 0.8176
Epoch 2: Training Loss = 0.5761, Validation Loss = 0.5170, Validation Accuracy = 0.8298
Epoch 3: Training Loss = 0.5534, Validation Loss = 0.4709, Validation Accuracy = 0.8548
Epoch 4: Training Loss = 0.5303, Validation Loss = 0.4289, Validation Accuracy = 0.8764
Epoch 5: Training Loss = 0.3354, Validation Loss = 0.4014, Validation Accuracy = 0.8846
Epoch 6: Training Loss = 0.4232, Validation Loss = 0.4308, Validation Accuracy = 0.8769
Epoch 7: Training Loss = 0.4327, Validation Loss = 0.4002, Validation Accuracy = 0.8860
Epoch 8: Training Loss = 0.4811, Validation Loss = 0.3615, Validation Accuracy = 0.8982
Epoch 9: Training Loss = 0.3377, Validation Loss = 0.3588, Validation Accuracy = 0.9016
Epoch 10: Training Loss = 0.4679, Validation Loss = 0.3673, Validation Accuracy = 0.8971


# Training using model.fit()

In [24]:
# Instantiate and compile the model
keras_fit_model = MLP()
keras_fit_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.01), loss=tf.keras.losses.CategoricalCrossentropy(from_logits=False), metrics=['accuracy'])

# Train using fit
keras_fit_model.fit(trainX, trainY, batch_size=64, epochs=10, validation_split=0.2)


Epoch 1/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.4757 - loss: 1.3600 - val_accuracy: 0.7745 - val_loss: 0.7205
Epoch 2/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.7763 - loss: 0.7035 - val_accuracy: 0.8049 - val_loss: 0.6328
Epoch 3/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7937 - loss: 0.6394 - val_accuracy: 0.8085 - val_loss: 0.6004
Epoch 4/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8046 - loss: 0.6118 - val_accuracy: 0.8192 - val_loss: 0.5744
Epoch 5/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8190 - loss: 0.5820 - val_accuracy: 0.8238 - val_loss: 0.5636
Epoch 6/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8250 - loss: 0.5614 - val_accuracy: 0.8227 - val_loss: 0.5628
Epoch 7/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8301 - loss: 0.5503 - val_accuracy: 0.8273 - val_loss: 0.5511
Epoch 8/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8306 - loss: 0.5509 - val_accuracy: 0.

# Model evaluation

In [22]:
def evaluate(model, testX):
    y_pred_prob = model(testX)
    y_pred = tf.argmax(y_pred_prob, axis=1)
    y_true = tf.argmax(testY, axis=1)

    acc = tf.reduce_mean(tf.cast(tf.equal(y_pred, y_true), tf.float32))
    return acc

In [23]:
print(f"Gradient Tape model accuracy: {evaluate(gradient_tape_model, testX) * 100:.2f}%")
print(f"Keras fit model accuracy: {evaluate(keras_fit_model, testX) * 100:.2f}%")

Gradient Tape model accuracy: 89.12%
Keras fit model accuracy: 85.59%
